# status

> Feedback and loading states: alerts, toasts, progress bars, spinners and skeletons

In [ ]:
#| default_exp status

These helpers return feedback and loading markup without running the underlying operation. You control the application state and decide when to add, update, or remove each component.

In [ ]:
from fasthtml.common import Div, Span, H3, P, Label, Template, unqid, Progress as _Progress
from fastcore.xml import FT
from fasthtml.core import FtResponse
from fastlucide.icons import X
from answerui.registry import hypelt
from answerui.controls import Btn
from answerui.surfaces import Card
from answerui.layout import Stack
from answerui.demo import example

In [ ]:
from fasthtml.common import Form
from answerui.demo import demo
from fastcore.test import *

## Notice

`_notice` builds the shared inner structure for `Alert` and `Toast`: a `Stack` of content beside an optional quiet icon close button marked `dismiss`. Shared CSS styles the flex row, surface, and expanding content column.


In [ ]:
def _notice(c, dismissible, **attrs):
    close = Btn(X(), icon=True, variant='quiet', aria_label='Close', dismiss=True) if dismissible else None
    return Div(Stack(*c), close, **attrs)

## Alert

`Alert` renders a message and optional supporting content on a tinted accent surface (`[alert]`). Pass `ui_tone` (`'warning'`, `'danger'`, `'success'`, or `'info'`) to choose a severity hue. With `dismissible=True`, clicking the close button removes the alert through the delegated `click` listener in `browser.js`. Because that listener is attached to `document`, alerts inserted later by htmx are dismissible without extra wiring.

In [ ]:
@hypelt(sel='[alert]')
def Alert(
    *c,                # The message and any supporting content
    dismissible=False, # Include a button that removes this alert
    **attrs,           # Attributes for the containing div, including ui_tone and role
):
    "An in-page message with optional dismissal"
    return _notice(c, dismissible, alert=True, **attrs)

Severity colour and screen-reader announcements are separate choices. `Alert` adds no `role` by default because a persistent warning need not interrupt a screen-reader user. For polite feedback, place an empty `role="status"` region in the page first, then insert or update the message inside it. Reserve `role="alert"` for urgent messages. Screen readers do not reliably announce a message when a single swap creates the live region with the message already inside it. See [ARIA live regions](https://developer.mozilla.org/en-US/docs/Web/Accessibility/ARIA/Guides/Live_regions).

In [ ]:
@example(Alert)
def _alert_example():
    return Stack(
        Alert('Your changes have been saved.', ui_tone='success', dismissible=True),
        Alert(H3('Storage is nearly full'), P('Remove unused files or choose a larger plan.'), ui_tone='warning'),
        Alert('The upload failed. Your original file has not changed.', ui_tone='danger', dismissible=True))

In [ ]:
Alert.demo()

```html
<div stack>
  <div alert ui-tone="success">
    <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button>  </div>
  <div alert ui-tone="warning">
    <div stack style="--gap: var(--space-2);">
      <h3>Storage is nearly full</h3>
      <p>Remove unused files or choose a larger plan.</p>
    </div>
  </div>
  <div alert ui-tone="danger">
    <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button>  </div>
</div>

```

Alerts have no auto-dismiss timer because important information must stay visible until the user reads or closes it.

## Toast

Place an empty `Toaster()` in the page before delivering messages to it. `Toaster` renders a polite live region (`role="status"`, `aria-live="polite"`) with `popover="manual"`. When a `[toaster]` mounts, `browser.js` calls `showPopover()`. Because it is a manual popover, the browser draws it above ordinary page content and leaves it open when the user clicks outside or presses Escape. Pass `toaster_position` (`'top-start'`, `'bottom-start'`, or `'bottom-end'`) to move it from the default top-end corner. A modal dialog opened later sits above the toaster. Put feedback needed during a modal interaction inside that dialog.

By default, `Toast` sets `--toast-duration: 5000ms` (controlled by `life`) for the `ui-toast-lifetime` CSS animation. While the pointer hovers over a toast or keyboard focus is inside it, CSS pauses that animation (`animation-play-state: paused`). When `animationend` fires for `ui-toast-lifetime`, `browser.js` removes the toast. Passing `life=None` omits `--toast-duration` to keep the toast visible until the user dismisses it. Under `prefers-reduced-motion: reduce`, `--motion` drops to `0ms` while `--toast-duration` stays unchanged.

In [ ]:
@hypelt(sel='[toaster]')
def Toaster(*c, **attrs):
    "The page's polite live region for toast deliveries"
    return Div(*c, popover='manual', role='status', aria_live='polite', aria_atomic='false',
        aria_relevant='additions text', toaster=True, **attrs)

In [ ]:
@hypelt(sel='[toast]')
def Toast(
    *c,               # The message
    life=5000,        # Unattended milliseconds before removal; None to stay until dismissed
    dismissible=True, # Include a dismiss button
    oob=False,        # Wrap for out-of-band delivery into the page's region
    **attrs,          # Attributes for the message, including ui_tone
):
    "A brief message for the toast region"
    if life is not None: attrs['style'] = f'--toast-duration: {life}ms; ' + attrs.get('style', '')
    toast = _notice(c, dismissible, toast=True, **attrs)
    return Div(toast, hx_swap_oob='beforeend:[toaster]') if oob else toast

Deliver a toast inside an out-of-band wrapper with `Div(Toast('Saved'), hx_swap_oob='beforeend:[toaster]')` or `Toast('Saved', oob=True)`. During a `beforeend` swap, htmx discards the outer element and appends its children. Wrapping the toast keeps the `[toast]` element, its attributes, and its dismiss button intact. Putting `hx_swap_oob` directly on `Toast` would make htmx discard the `[toast]` element and append only its inner nodes. See [out-of-band swaps](https://htmx.org/attributes/hx-swap-oob/).

The example below stores the out-of-band wrapper in a `template` element whose HTML the button's `hx-on:click` handler passes to [htmx.swap](https://htmx.org/api/#swap). This demonstrates delivery without a server or a registered behavior. In an app, return the wrapper directly from a route with no inline script. Click the button to deliver both messages, then try hovering or focusing the timed toast.

In [ ]:
@example(Toast)
def _toast_example():
    id = unqid()
    response = Div(Toast('Your changes have been saved.', ui_tone='success'),
        Toast('The upload failed. Your original file has not changed.', life=None, ui_tone='danger'),
        hx_swap_oob='beforeend:[toaster]')
    return Div(Btn('Deliver notifications', variant='secondary',
        hx_on_click=f"htmx.swap(this, document.getElementById('{id}').innerHTML, {{swapStyle: 'none'}})"),
        Template(response, id=id), Toaster())

In [ ]:
Toast.demo(height='14rem')

```html
<div>
<button type="button" btn btn-variant="secondary" hx-on-click="htmx.swap(this, document.getElementById('_ZzRR0yYaRSK9P47PhmKSmg').innerHTML, {swapStyle: 'none'})">Deliver notifications</button><template id="_ZzRR0yYaRSK9P47PhmKSmg">    <div hx-swap-oob="beforeend:[toaster]">
      <div toast ui-tone="success" style="--toast-duration: 5000ms; ">
        <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button>      </div>
      <div toast ui-tone="danger">
        <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button>      </div>
    </div>
</template>  <div popover="manual" role="status" aria-live="polite" aria-atomic="false" aria-relevant="additions text" toaster></div>
</div>

```

### Queued toasts

A redirect response contains no HTML for a toast. An htmx route returns a partial fragment rather than a full page with a `Toaster`. In both cases, call `add_toast(sess, msg, life=5000, **attrs)` to queue the message in the session under `toasts_key` (`'aui_toasts'`), passing severity as an attribute such as `ui_tone='success'`.

On a full-page route, render queued toasts inside `Toaster(*pop_toasts(sess))`. On a fragment route, include `toasts_oob(sess)` in the response for htmx to swap into the existing `[toaster]`. Call `setup_toasts(app)` to register `toast_after` on `app.after` and inject `toasts_oob(sess)` automatically on every markup or empty response. `toast_after` does nothing when the queue is empty, as happens after a full-page route calls `pop_toasts(sess)`.

In [ ]:
toasts_key = 'aui_toasts'

def add_toast(sess, msg, life=5000, **attrs):
    "Queue `msg` in session `sess` for the next page or fragment, with its severity in `attrs`"
    sess.setdefault(toasts_key, []).append((msg, life, attrs))

def pop_toasts(sess):
    "The toasts queued in session `sess`, as `Toast`s, clearing the queue"
    return [Toast(m, life=l, **a) for m,l,a in sess.pop(toasts_key, [])]

def toasts_oob(sess):
    "The toasts queued in session `sess`, in an out-of-band wrapper for delivery beside a fragment, clearing the queue; None when nothing is queued"
    toasts = pop_toasts(sess)
    return Div(*toasts, hx_swap_oob='beforeend:[toaster]') if toasts else None

`toast_after` appends `toasts_oob(sess)` to `req.injects` when a route returns markup (`tuple`, `FT`, or `FtResponse`) or an empty response (for routes that only queue a toast). After a redirect or file response, queued toasts stay in the session for the next page:

In [ ]:
def toast_after(resp, req, sess):
    "FastHTML `after` hook that adds the queued toasts out of band to a markup or empty response"
    if sess.get(toasts_key) and (not resp or isinstance(resp, (tuple, FT, FtResponse))): req.injects.append(toasts_oob(sess))

def setup_toasts(app):
    "Deliver queued toasts out of band with every markup or empty response of `app`"
    app.after.append(toast_after)

Queue two messages as a route does and pop them with `toasts_oob`, clearing `sess`:

In [ ]:
sess = {}
add_toast(sess, 'Your changes have been saved.', ui_tone='success')
add_toast(sess, 'The upload failed. Your original file has not changed.', life=None, ui_tone='danger')
oob = toasts_oob(sess)
test_eq(sess, {})
test_eq([t.attrs.get('ui-tone') for t in oob.children], ['success', 'danger'])
oob

<div hx-swap-oob="beforeend:[toaster]"><div toast ui-tone="success" style="--toast-duration: 5000ms; "><div stack style="--gap: var(--space-2);">Your changes have been saved.</div><button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button></div><div toast ui-tone="danger"><div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div><button type="button" btn btn-variant="quiet" btn-icon aria-label="Close" dismiss><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button></div></div>

## Progress

`Progress` wraps the native `<progress>` element (`progress[progress]`). Pass `value` and `max` when the completed fraction is known, or omit `value` for an indeterminate bar. The `[progress]` CSS rule sets only `inline-size: 100%`, `block-size: var(--space-2)`, and `accent-color: var(--color-accent)` without changing the browser's native rendering or animation. Wrap the bar in a `Label` describing the operation.

In [ ]:
@hypelt(sel='progress[progress]')
def Progress(*c, **attrs):
    "Native progress; omit value when the amount completed is unknown"
    return _Progress(*c, progress=True, **attrs)

A labelled bar for a known amount of work, and an indeterminate one for an unknown amount:


In [ ]:
@example(Progress)
def _progress_example():
    return Stack(Label('Importing files', Progress(value=40, max=100), stack=True, cls='gap-1'),
        Label('Waiting for the server', Progress(), stack=True, cls='gap-1'))


In [ ]:
Progress.demo()

```html
<div stack>
<label stack style="--gap: var(--space-3)">Importing files<progress progress value="40" max="100"></progress></label><label stack style="--gap: var(--space-3)">Waiting for the server<progress progress></progress></label></div>

```

## Spinner

`Spinner` renders a `Span` (`[spinner]`) with a CSS ring (`::before`) beside visible status text (`label`, defaulting to `'Loading...'`). The ring uses the `ui-spin` animation with duration `calc(var(--motion) * 5)`. Under `prefers-reduced-motion: reduce`, the animation stops while the label remains visible. As with `Alert`, add a live-region `role` only when screen readers should announce the status.

In [ ]:
@hypelt(sel='[spinner]')
def Spinner(label='Loading...', **attrs):
    "A small spinning ring with visible text explaining the wait"
    return Span(label, spinner=True, **attrs)

In [ ]:
@example(Spinner)
def _spinner_example():
    return Spinner('Checking the connection…')

In [ ]:
Spinner.demo()

```html
<span spinner>Checking the connection…</span>
```

## Skeleton

`Skeleton` renders an unanimated block placeholder (`Span` with `[skeleton]` and `aria-hidden="true"`) that reserves layout space before content arrives. By default it spans `inline-size: 100%` and `block-size: 1em` with `var(--n-1)` background and `var(--radius-1)` corners. Override those dimensions with CSS such as `style`. Because `aria-hidden="true"` hides the placeholder from the accessibility tree, provide a separate visible loading message for screen readers.

In [ ]:
@hypelt(sel='[skeleton]')
def Skeleton(**attrs):
    "A decorative placeholder for content that has not loaded"
    return Span(skeleton=True, aria_hidden='true', **attrs)

In [ ]:
@example(Skeleton)
def _skeleton_example():
    return Stack(P('Loading the project summary…'),
        Card(Skeleton(style='inline-size: 40%; block-size: 1.5em'), Skeleton(), Skeleton(style='inline-size: 75%'),
            aria_busy='true', stack=True))

In [ ]:
Skeleton.demo()

```html
<div stack>
  <p>Loading the project summary…</p>
  <section card aria-busy="true" stack>
<span skeleton aria-hidden="true" style="inline-size: 40%; block-size: 1.5em"></span><span skeleton aria-hidden="true"></span><span skeleton aria-hidden="true" style="inline-size: 75%"></span>  </section>
</div>

```

The loading message is outside the busy region. When the content arrives, replace the placeholders and clear `aria-busy`, or replace the whole card. `Skeleton` does not make these changes itself.